In [0]:

%run ./01-config

In [0]:
# Configuration paths for data storage.
# These assume `base_dir_data` is defined elsewhere (e.g., as a notebook widget, environment variable, or config).
# - `landing_zone`: Directory for raw, unprocessed input files (not used in this snippet but part of the broader pipeline).
# - `test_data_dir`: Directory containing pre-generated static test or reference datasets (e.g., for demo or validation).
landing_zone = base_dir_data + "/raw"
test_data_dir = base_dir_data + "/test_data"

In [0]:
# Configuration paths for data storage.
# These assume `base_dir_data` is defined elsewhere (e.g., as a notebook widget, environment variable, or config).
# - `landing_zone`: Directory for raw, unprocessed input files (not used in this snippet but part of the broader pipeline).
# - `test_data_dir`: Directory containing pre-generated static test or reference datasets (e.g., for demo or validation).
landing_zone = base_dir_data + "/raw"
test_data_dir = base_dir_data + "/test_data"


# Loads reference date dimension data into the `date_lookup` table from a static JSON file.
# This table typically supports time-based analytics (e.g., reporting by week, month, or custom periods like "week_part").
# Uses `INSERT OVERWRITE` to replace all existing data—ensuring idempotency and consistency.
# The source file is expected to be a JSON directory (Delta/Parquet-style layout) at:
#   {test_data_dir}/6-date-lookup.json/
def load_date_lookup(catalog, db_name):
    print(f"Loading date_lookup table...", end='')        
    spark.sql(f"""INSERT OVERWRITE TABLE {catalog}.{db_name}.date_lookup 
            SELECT date, week, year, month, dayofweek, dayofmonth, dayofyear, week_part 
            FROM json.`{test_data_dir}/6-date-lookup.json/`""")
    print("Done")


# Orchestrates the loading of all historical/reference datasets.
# Currently only loads the `date_lookup` table, but structured to support additional historical loads in the future.
# Measures and reports total execution time for observability and performance tracking.
def load_history(catalog, db_name):
    import time
    start = int(time.time())
    print(f"\nStarting historical data load ...")
    load_date_lookup(catalog, db_name)
    print(f"Historical data load completed in {int(time.time()) - start} seconds")


# Validates that a given table contains exactly the expected number of records.
# Used to confirm data integrity after ingestion or transformation.
# Compares actual row count (via Spark DataFrame `.count()`) against a known expected value.
# Raises an AssertionError if counts don’t match—useful in testing or CI/CD pipelines.
# Formats large numbers with commas for readability (e.g., "1,234").
def assert_count(catalog, db_name, table_name, expected_count):
    print(f"Validating record counts in {table_name}...", end='')
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").count()
    assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}" 
    print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")
        

# Validates the correctness of historical data loads by checking row counts.
# Currently only validates the `date_lookup` table, which is expected to contain 365 records
# (likely representing one full non-leap year of daily records).
# Measures and reports validation duration.
# This function is typically called after `load_history()` to ensure data was loaded correctly.
def validate(catalog, db_name):
    import time
    start = int(time.time())
    print(f"\nStarting historical data load validation...")
    assert_count(catalog, db_name, "date_lookup", 365)
    print(f"Historical data load validation completed in {int(time.time()) - start} seconds")
